##### Stage 1 — Data understanding & memory engineering. ✅ Done. You worked through the customer/statement structure, the class imbalance and the 20× downsampling correction, the metric's business meaning, memory-safe loading (the whole CSV-to-Parquet saga), and then EDA: missingness by family, statement-count distribution, and categorical cardinality.

##### Stage 2 — Implement the competition metric (M) yourself. Build M as a Python function — the normalized Gini plus top-4% capture, with the 20× negative weighting — and unit-test it against toy inputs where you know the right answer, before any modeling. The point: you must trust your own scoring before you can trust any model's validation score.

##### Stage 3 — A dead-simple baseline. Last statement per customer + LightGBM, end to end, just to get a working pipeline and a first CV/leaderboard sanity check. Deliberately crude — the goal is a complete loop, not a good score.

##### Stage 4 — Real feature engineering. Per-customer aggregations across the 13 statements: mean/std/min/max/last, deltas, lag features. This is where most of the actual score lives in this competition, more than model choice — and where all your EDA findings (missingness handling, the −1 sentinels, categorical vs numeric treatment, statement count as a feature) get used.

##### Stage 5 — Model tuning. LightGBM/XGBoost specifics for this metric — custom eval function, categorical handling, early stopping, hyperparameters.

##### Stage 6 — Cross-validation strategy. A validation scheme that doesn't leak and actually correlates with the leaderboard. Critical, and easy to get subtly wrong.

##### Stage 7 — (Optional) Sequence modeling. A GRU or Transformer over the raw 13-month history — plays directly to your PyTorch/CV background, and is what several top solutions did alongside the gradient-boosted trees.

##### Stage 8 — Ensembling. Combine models, and only at the very end, read the top write-ups to see what you missed.

### 1. Creating the paths for the files

In [1]:
import os

BASE_PATH = "/kaggle/input/datasets/raddar/amex-data-integer-dtypes-parquet-format"
train_file = os.path.join(BASE_PATH, "train.parquet")
test_file = os.path.join(BASE_PATH, "test.parquet")

### 2. Importing Numpy and Pandas

In [2]:
import numpy as np
import pandas as pd

### 3. Loading the dataset and previewing the data

In [3]:
train_data = pd.read_parquet(train_file)

row_count = train_data.shape[0]

print(row_count)
print(train_data["customer_ID"].nunique())
train_data.info(memory_usage="deep")

5531451
458913
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5531451 entries, 0 to 5531450
Columns: 190 entries, customer_ID to D_145
dtypes: float32(93), int16(9), int8(86), object(2)
memory usage: 3.3 GB


## Stage 1 - EDA

#### # Missingness by prefix group

In [4]:
missing_pct = train_data.isnull().mean() * 100

miss_df = pd.DataFrame({"column": missing_pct.index, "missing_pct": missing_pct.values})

miss_df = miss_df[~miss_df["column"].isin(["customer_ID", "S_2"])]

miss_df["prefix"] = miss_df["column"].str[0]

prefix_summary = miss_df.groupby("prefix")["missing_pct"].agg(["mean", "max", "min", "count"])

print("Missingness by prefix group:\n")
print(prefix_summary.round(2))

# Also surface the worst individual offenders overall
print("\nTop 15 most-missing columns:\n")
print(miss_df.sort_values("missing_pct", ascending=False).head(15).to_string(index=False))


Missingness by prefix group:

         mean    max  min  count
prefix                          
B        8.74  99.39  0.0     40
D       11.96  99.89  0.0     96
P        2.09   5.45  0.0      3
R        0.08   2.33  0.0     28
S        5.53  53.04  0.0     21

Top 15 most-missing columns:

column  missing_pct prefix
  D_88    99.891457      D
 D_110    99.433530      D
  B_39    99.391986      B
  D_73    98.990211      D
  B_42    98.707789      B
 D_134    96.480146      D
  B_29    93.104594      B
 D_132    90.191055      D
  D_76    88.746226      D
  D_42    85.694278      D
 D_142    82.926577      D
  D_53    73.842921      D
  D_50    56.809723      D
  B_17    56.722874      B
 D_105    54.622756      D


In [5]:
# How many rows (monthly statements) each customer has
stmt_counts = train_data.groupby("customer_ID").size()

print("\nStatement-count distribution (how many customers have N statements):\n")
print(stmt_counts.value_counts().sort_index())

# What fraction have the full 13 vs fewer?
full_13 = (stmt_counts == 13).mean() * 100
print(f"\n% of customers with full 13 statements: {full_13:.2f}%")
print(f"% with fewer than 13: {100 - full_13:.2f}%")
print(f"\nSummary stats of statement counts:\n{stmt_counts.describe().round(2)}")


Statement-count distribution (how many customers have N statements):

1       5120
2       6098
3       5778
4       4673
5       4671
6       5515
7       5198
8       6110
9       6411
10      6721
11      5961
12     10623
13    386034
Name: count, dtype: int64

% of customers with full 13 statements: 84.12%
% with fewer than 13: 15.88%

Summary stats of statement counts:
count    458913.00
mean         12.05
std           2.61
min           1.00
25%          13.00
50%          13.00
75%          13.00
max          13.00
dtype: float64


In [6]:
cat_cols = ["B_30", "B_38", "D_114", "D_116", "D_117",
            "D_120", "D_126", "D_63", "D_64", "D_66", "D_68"]

print("\nCategorical columns — dtype, unique count, and the actual values:\n")
for col in cat_cols:
    n_unique = train_data[col].nunique(dropna=True)
    dtype = train_data[col].dtype
    uniques = sorted(train_data[col].dropna().unique().tolist())
    # Trim the printout if a column has many uniques
    shown = uniques if len(uniques) <= 6 else uniques[:6] + ["..."]
    print(f"{col:6} | dtype={str(dtype):8} | n_unique={n_unique:3} | values={shown}")


Categorical columns — dtype, unique count, and the actual values:

B_30   | dtype=int8     | n_unique=  4 | values=[-1, 0, 1, 2]
B_38   | dtype=int8     | n_unique=  8 | values=[-1, 1, 2, 3, 4, 5, '...']
D_114  | dtype=int8     | n_unique=  3 | values=[-1, 0, 1]
D_116  | dtype=int8     | n_unique=  3 | values=[-1, 0, 1]
D_117  | dtype=int8     | n_unique=  8 | values=[-1, 0, 2, 3, 4, 5, '...']
D_120  | dtype=int8     | n_unique=  3 | values=[-1, 0, 1]
D_126  | dtype=int8     | n_unique=  4 | values=[-1, 0, 1, 2]
D_63   | dtype=int8     | n_unique=  6 | values=[0, 1, 2, 3, 4, 5]
D_64   | dtype=int8     | n_unique=  5 | values=[-1, 0, 1, 2, 3]
D_66   | dtype=int8     | n_unique=  3 | values=[-1, 0, 1]
D_68   | dtype=int8     | n_unique=  8 | values=[-1, 0, 1, 2, 3, 4, '...']


## Stage 2 - Building the competition metric

In [14]:
def top_four_percent(y_true, y_pred):
    # 1. Sort by prediction descending
    sorted_indices = np.argsort(y_pred)[::-1] # slicing works like this [start:stop:step]

    # 2. Reorder y_true and y_pred using that sorted order
    y_true_sorted = y_true[sorted_indices]

    # 3. Assign weights: 20 for label 0, 1 for label 1
    weights = np.where(y_true_sorted == 0, 20, 1)  # np.where(condition, value_if_true, value_if_false)

    # 4. Calculate cumulative weights
    cumulative_weights = np.cumsum(weights)
    total_weight = cumulative_weights[-1]
    
    # 5. Calculate cutoff = 4% of total weight
    cutoff = 0.04 * total_weight

    # 6. Select rows where cumulative weight <= cutoff
    selected_rows = cumulative_weights <= cutoff

    # 7. Return captured positives / total positives
    captured_positives = np.sum(y_true_sorted[selected_rows] == 1)
    total_positives = np.sum(y_true == 1)

    return captured_positives / total_positives


y_true = np.array([1, 1, 1, 0, 0, 0])
y_pred = np.array([0.99, 0.95, 0.90, 0.40, 0.20, 0.10])

top_four_percent(y_true, y_pred)

np.float64(0.6666666666666666)